<a href="https://colab.research.google.com/github/rafiul254/Flyrank_AI-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafiul254/Flyrank_AI-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!rm -f content_refresh_anonymized.csv
!wget -q "https://github.com/rafiul254/Flyrank_AI-ML-internship/raw/refs/heads/main/data/raw/content_refresh_anonymized.csv" -O content_refresh_anonymized.csv

import os
size = os.path.getsize("content_refresh_anonymized.csv")
print(f"File size: {size:,} bytes")
print(f"File exists: {size > 1000}")

File size: 6,727,670 bytes
File exists: True


In [20]:
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")
print(list(df.columns))

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("content_refresh_anonymized.csv")

# Create label from trend_direction (label only — never a feature)
df["is_declining_label"] = (df["trend_direction"] == "declining").astype(int)

df_pos = df[df["avg_position"] > 0].copy()
df_pos["is_declining_label"] = (df_pos["trend_direction"] == "declining").astype(int)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Rows with valid avg_position: {df_pos.shape[0]:,}")
print(f"Declining pages: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean()*100:.1f}%)\n")

# ══════════════════════════════════════════════════
# SIGNAL 1 — Staleness (days_since_last_update)
# ══════════════════════════════════════════════════
print("="*55)
print("SIGNAL 1: Staleness (days_since_last_update)")
print("="*55)

bins   = [0, 30, 90, 180, 365, 99999]
labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

sig1 = (df
    .groupby("staleness_bucket", observed=True)
    .agg(n=("is_declining_label","count"),
         pct_declining=("is_declining_label","mean"))
    .reset_index()
)
sig1["pct_declining_%"] = (sig1["pct_declining"] * 100).round(1)
sig1 = sig1.drop(columns="pct_declining")
print(sig1.to_string(index=False))
print(f"\nTotal n: {sig1['n'].sum():,}")

old   = sig1.loc[sig1["staleness_bucket"]=="365d+","pct_declining_%"].values[0]
fresh = sig1.loc[sig1["staleness_bucket"]=="0-30d","pct_declining_%"].values[0]
print("\nSIGNAL 1 VERDICT →", end=" ")
if old > fresh + 3:
    print("CONFIRMED — stale pages decline meaningfully more than fresh ones.")
elif old < fresh - 3:
    print("OPPOSITE — fresh pages decline more. Re-examine rule.")
else:
    print("MIXED — no strong trend across staleness buckets.")

# ══════════════════════════════════════════════════
# SIGNAL 2 — CTR vs Position bucket
# ══════════════════════════════════════════════════
print("\n" + "="*55)
print("SIGNAL 2: CTR vs Position bucket")
print("="*55)

df_pos["pos_bucket"] = pd.cut(
    df_pos["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, 9999],
    labels=["1-3","4-5","6-10","11-20","21-50","50+"]
)

sig2 = (df_pos
    .groupby("pos_bucket", observed=True)
    .agg(n=("ctr","count"),
         avg_ctr=("ctr","mean"),
         median_ctr=("ctr","median"))
    .reset_index()
)
sig2["avg_ctr"]    = sig2["avg_ctr"].round(4)
sig2["median_ctr"] = sig2["median_ctr"].round(4)
print(sig2.to_string(index=False))
print(f"\nTotal n: {sig2['n'].sum():,}")
print("Note: CTR is x100 — e.g. 0.76 means 0.76%, not 76%")

top_ctr = sig2.iloc[0]["avg_ctr"]
bot_ctr = sig2.iloc[-2]["avg_ctr"]
print("\nSIGNAL 2 VERDICT →", end=" ")
if top_ctr > bot_ctr * 1.5:
    print("CONFIRMED — CTR clearly decreases as position worsens.")
elif top_ctr < bot_ctr:
    print("OPPOSITE — unexpected. Investigate data quality.")
else:
    print("MIXED — CTR pattern is not strongly monotonic.")

Loaded: 30,000 rows × 45 columns
Rows with valid avg_position: 28,795
Declining pages: 0 (0.0%)

SIGNAL 1: Staleness (days_since_last_update)
staleness_bucket     n  pct_declining_%
           0-30d 20480              0.0
          31-90d   175              0.0
         91-180d  9171              0.0
        181-365d   169              0.0
           365d+     5              0.0

Total n: 30,000

SIGNAL 1 VERDICT → MIXED — no strong trend across staleness buckets.

SIGNAL 2: CTR vs Position bucket
pos_bucket    n  avg_ctr  median_ctr
       1-3 1141   2.7143        0.00
       4-5 2782   1.1048        0.23
      6-10 9060   0.5117        0.14
     11-20 7273   0.3234        0.10
     21-50 7225   0.2223        0.03
       50+ 1314   0.1508        0.00

Total n: 28,795
Note: CTR is x100 — e.g. 0.76 means 0.76%, not 76%

SIGNAL 2 VERDICT → CONFIRMED — CTR clearly decreases as position worsens.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
df2 = pd.read_csv("content_refresh_anonymized.csv")
df2 = df2[df2["avg_position"] > 0].copy()
df2["is_declining_label"] = (df2["trend_direction"] == "declining").astype(int)
print(f"Working dataset: {df2.shape[0]:,} rows")

# Position-bucket CTR median
df2["pos_bucket"] = pd.cut(
    df2["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, 9999],
    labels=["1-3","4-5","6-10","11-20","21-50","50+"]
)
ctr_medians = df2.groupby("pos_bucket", observed=True)["ctr"].median()
df2["ctr_bucket_median"] = df2["pos_bucket"].map(ctr_medians)
df2["below_median_ctr"]  = (df2["ctr"] < df2["ctr_bucket_median"]).astype(int)

# Binary gates
df2["stale"]   = (df2["days_since_last_update"] >= 180).astype(int)
df2["visible"] = (df2["impressions_90d"] >= 500).astype(int)

# Score
df2["score"] = (
    df2["stale"] * df2["visible"] * df2["impressions_90d"] * (1 + df2["below_median_ctr"])
)

# Reason codes
def reason_code(row):
    if   row["stale"] and row["visible"] and row["below_median_ctr"]: return "stale_visible_low_ctr"
    elif row["stale"] and row["visible"]:                              return "stale_visible_ok_ctr"
    elif not row["stale"]:                                             return "not_stale"
    else:                                                              return "not_visible"

df2["reason_code"] = df2.apply(reason_code, axis=1)

action_map = {
    "stale_visible_low_ctr": "REFRESH_PRIORITY",
    "stale_visible_ok_ctr" : "REFRESH_CONSIDER",
    "not_stale"            : "HOLD",
    "not_visible"          : "HOLD_LOW_VOLUME"
}
df2["action"] = df2["reason_code"].map(action_map)

df2 = df2.sort_values("score", ascending=False).reset_index(drop=True)
df2["rank"] = df2.index + 1

import os
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["rank","content_id","client_id","score","reason_code","action",
            "days_since_last_update","impressions_90d","ctr","avg_position"]
out_path = "work/outputs/baseline_action_score.csv"
df2[out_cols].to_csv(out_path, index=False)

print(f"✅  Written: {out_path}")
print(f"    Rows: {df2.shape[0]:,}\n")
print("Action distribution:")
print(df2["action"].value_counts().to_string())
print("\nTop 5 preview:")
print(df2[out_cols].head().to_string(index=False))

Working dataset: 28,795 rows
✅  Written: work/outputs/baseline_action_score.csv
    Rows: 28,795

Action distribution:
action
HOLD                28637
HOLD_LOW_VOLUME       141
REFRESH_CONSIDER       14
REFRESH_PRIORITY        3

Top 5 preview:
 rank           content_id         client_id  score           reason_code           action  days_since_last_update  impressions_90d  ctr  avg_position
    1 content_cf56e2e2e282 client_7f2253d7e2  61678  stale_visible_ok_ctr REFRESH_CONSIDER                     194            61678 0.15          19.7
    2 content_7368877ea310 client_7f2253d7e2  59472  stale_visible_ok_ctr REFRESH_CONSIDER                     194            59472 0.13          24.8
    3 content_1bfaa38ff26c client_7f2253d7e2  25715  stale_visible_ok_ctr REFRESH_CONSIDER                     194            25715 0.23          22.2
    4 content_5feee3994adb client_7f2253d7e2  15624 stale_visible_low_ctr REFRESH_PRIORITY                     194             7812 0.01          39.0

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [23]:
top20 = df2[out_cols].head(20).copy()

print("TOP-20 RANKED QUEUE\n")
print(f"{'Rank':<5} {'Action':<20} {'Why it is here':<50} {'What would make it wrong'}")
print("-"*130)

for _, row in top20.iterrows():
    rank   = int(row["rank"])
    action = row["action"]
    days   = int(row["days_since_last_update"])
    imp    = int(row["impressions_90d"])
    ctr    = row["ctr"]
    pos    = round(row["avg_position"], 1)
    rc     = row["reason_code"]

    why = f"{days}d old | {imp:,} impr | CTR={ctr:.3f} | pos={pos}"

    if rc == "stale_visible_low_ctr":
        wrong = "CTR low due to SERP feature stealing clicks — not fixable by refresh"
    elif rc == "stale_visible_ok_ctr":
        wrong = "Page intentionally evergreen; staleness is by design"
    else:
        wrong = "Threshold borderline — slight change would drop it out"

    print(f"{rank:<5} {action:<20} {why:<50} {wrong}")

TOP-20 RANKED QUEUE

Rank  Action               Why it is here                                     What would make it wrong
----------------------------------------------------------------------------------------------------------------------------------
1     REFRESH_CONSIDER     194d old | 61,678 impr | CTR=0.150 | pos=19.7      Page intentionally evergreen; staleness is by design
2     REFRESH_CONSIDER     194d old | 59,472 impr | CTR=0.130 | pos=24.8      Page intentionally evergreen; staleness is by design
3     REFRESH_CONSIDER     194d old | 25,715 impr | CTR=0.230 | pos=22.2      Page intentionally evergreen; staleness is by design
4     REFRESH_PRIORITY     194d old | 7,812 impr | CTR=0.010 | pos=39.0       CTR low due to SERP feature stealing clicks — not fixable by refresh
5     REFRESH_CONSIDER     193d old | 13,299 impr | CTR=0.490 | pos=10.5      Page intentionally evergreen; staleness is by design
6     REFRESH_PRIORITY     194d old | 4,590 impr | CTR=0.000 | pos=31.0   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [24]:
FORBIDDEN     = ["trend_direction","trend_pct","is_declining_label",
                 "content_id","client_id"]
USED_FEATURES = ["days_since_last_update","impressions_90d","ctr","avg_position"]

print("Forbidden columns in feature set:")
leaks = [c for c in FORBIDDEN if c in USED_FEATURES]
print(f"  {'⚠️  LEAK: ' + str(leaks) if leaks else '✅  None — clean.'}\n")

print("Features used:")
for f in USED_FEATURES:
    print(f"  • {f}: min={df2[f].min():.2f}, max={df2[f].max():.2f}")

future_cols = [c for c in df2.columns if "next_" in c or "future" in c]
print(f"\nFuture-window columns: {future_cols if future_cols else 'None — clean.'}")

# Precision@K
print("\n── Precision@K ──────────────────────────────────────────")
labels_arr = df2["is_declining_label"].values
scores_arr = df2["score"].values
base_rate  = labels_arr.mean()
print(f"Base rate: {base_rate*100:.1f}%\n")

order = np.argsort(-scores_arr)
for k in [10, 20, 50, 100]:
    p    = labels_arr[order[:k]].mean()
    lift = p / base_rate if base_rate > 0 else 0
    print(f"Precision@{k:<4}: {p*100:.1f}%  (base={base_rate*100:.1f}%, lift={lift:.1f}x)")

Forbidden columns in feature set:
  ✅  None — clean.

Features used:
  • days_since_last_update: min=1.00, max=373.00
  • impressions_90d: min=1.00, max=517715.00
  • ctr: min=0.00, max=100.00
  • avg_position: min=0.10, max=245.00

Future-window columns: None — clean.

── Precision@K ──────────────────────────────────────────
Base rate: 0.0%

Precision@10  : 0.0%  (base=0.0%, lift=0.0x)
Precision@20  : 0.0%  (base=0.0%, lift=0.0x)
Precision@50  : 0.0%  (base=0.0%, lift=0.0x)
Precision@100 : 0.0%  (base=0.0%, lift=0.0x)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.